In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import time
import torch
from collections import defaultdict,deque

In [5]:
# model = YOLO("../../runs/detect/Custom_Vehicle_Detector_v1-4/weights/best.pt")
model = YOLO("yolov8n.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
cap = cv2.VideoCapture("video.mp4")
if cap.isOpened():
    print("yes")
else:
    raise FileNotFoundError("file not found")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_video = cap.get(cv2.CAP_PROP_FPS) or 30

output = cv2.VideoWriter(
    "output/track1.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps_video,
    (width, height)
)

yes


In [7]:
# results = model.track(
#     "video.mp4",
#     persist=True,
#     device=device,
#     tracker="bytetrack.yaml"
# )

# for result in results:
#     frame = result.plot()  
#     cv2.imshow("Frame", frame)

#     if cv2.waitKey(30) & 0xFF == ord('q'):
#         break

# cv2.destroyAllWindows()
# print(cap.isOpened())

In [8]:
vehicle_classes = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
person_class = 0
track_history = defaultdict(lambda: deque(maxlen=20))
unique_ids = set()
frame_idx = 0

# danger zone: bottom 20% of frame
zone_x1 = 0
zone_y1 = int(height * 0.20)
zone_x2 = width
zone_y2 = int(height* 0.35)

while cap.isOpened():
    start_time = time.time()
    success,frame = cap.read()
    if not success:
        break
    
    results = model.track(
        frame,
        persist=True,
        device=device,
        tracker="bytetrack.yaml",
        conf = 0.25,
        iou=0.5
    )
    
    annotated = frame.copy()

    # draw danger zone
    cv2.rectangle(annotated, (zone_x1, zone_y1), (zone_x2, zone_y2), (0, 255, 255), 2)
    cv2.putText(annotated, "DANGER ZONE", (zone_x1, zone_y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    r = results[0]

    if r.boxes is not None and r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)

        for box, cls_id, conf, track_id in zip(boxes, cls_ids, confs, track_ids):
            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2
            cy = y2

            label = r.names.get(cls_id, str(cls_id))
            is_person = cls_id == person_class
            in_danger = is_person and (zone_x1 <= cx <= zone_x2) and (zone_y1 <= cy <= zone_y2)

            if cls_id in vehicle_classes:
                unique_ids.add(track_id)

            color = (0, 0, 255) if in_danger else (0, 255, 0)

            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                annotated,
                f"{label.capitalize()} #{track_id} {conf:.2f}",
                (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

            if in_danger:
                cv2.putText(
                    annotated,
                    "PEDESTRIAN ALERT",
                    (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.0,
                    (0, 0, 255),
                    3
                )

            # trail
            track_history[track_id].append((cx, cy))
            pts = list(track_history[track_id])
            for p1, p2 in zip(pts, pts[1:]):
                cv2.line(annotated, p1, p2, (255, 255, 0), 2)


    car_count = sum(1 for c in cls_ids if c == 2) if r.boxes is not None and r.boxes.id is not None else 0
    person_count = sum(1 for c in cls_ids if c == 0) if r.boxes is not None and r.boxes.id is not None else 0
    motor_count = sum(1 for c in cls_ids if c == 3) if r.boxes is not None and r.boxes.id is not None else 0

    cv2.putText(annotated, f"Cars: {car_count}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(annotated, f"People: {person_count}", (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(annotated, f"Motorcycles: {motor_count}", (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(annotated, f"Unique Vehicles: {len(unique_ids)}", (20, 150),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    output.write(annotated)
    # cv2.imshow("video",annotated)
    # if cv2.waitKey(1) & 0xFF == ord("q"):
    #     break
    frame_idx += 1
    
cap.release()
output.release()
cv2.destroyAllWindows()



0: 384x640 1 person, 3 cars, 74.6ms
Speed: 82.9ms preprocess, 74.6ms inference, 14.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 7.2ms
Speed: 1.5ms preprocess, 7.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 8.4ms
Speed: 1.7ms preprocess, 8.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 8.1ms
Speed: 1.2ms preprocess, 8.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 7.7ms
Speed: 1.2ms preprocess, 7.7ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 10.8ms
Speed: 1.1ms preprocess, 10.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 traffic light, 7.6ms
Speed: 1.4ms preprocess, 7.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 traffic light, 7.9ms
Speed: 1.4ms preproc

In [9]:
# to view the file view it manually for some reason the output wont show in vs code